# Step 11 · attack the FINAL system (round-2 DeBERTa + registry-relative calibration) + seed repeats
1. `adv_train.py` (seed 42): re-creates the round-2 ADV DeBERTa (~6 min).
2. `attack_whitebox.py --set whitebox.calibrated=true`: the white-box attack now targets each server's **calibrated**
   threshold (rel threshold + the server's anchor), 30% budget, 100 steps, then the agent check (~40–60 min).
3. `calibrate_adv.py` with seeds 43 and 44: error bars for the final-system false-alarm and detection numbers (~15 min).
**Settings:** GPU **T4 x2** · Internet **On** · Secrets **Hugging Face** + **mcp-git** · Save Version → Save & Run All.


In [ ]:
# ---- settings for this run ----
REPO = "github.com/Arn15/mcp-poison-detect"
BRANCH = "main"
SRC_TRAIN = "[results/wb1/whitebox.jsonl,results/wb1e5/whitebox.jsonl,results/wb3_base_b30_deb/whitebox.jsonl,results/wb3_adv_b30_deb/whitebox.jsonl]"
SRC_CAL = "[results/wb1,results/wb1e5,results/wb3_base_b30_deb,results/wb3_adv_b30_deb]"
ATTACK = "whitebox.max_frac=0.30 whitebox.steps=100 whitebox.calibrated=true"
GPU_DEPS = True
MIN_COMMIT_FILE = "notebooks/experiments/kaggle_final_attack.ipynb"


In [ ]:
import os, subprocess, base64
from kaggle_secrets import UserSecretsClient
sec = UserSecretsClient()
def secret(k):
    try: return sec.get_secret(k)
    except Exception: return None
HF_SECRET, GH_SECRET = "Hugging Face", "mcp-git"   # names of your Kaggle secrets
os.environ["HF_TOKEN"] = secret(HF_SECRET) or ""
gh = secret(GH_SECRET)
assert gh, "Secret 'mcp-git' not attached to this notebook (Add-ons -> Secrets)"
WORK = "/kaggle/working"

# Token goes in an HTTP header for this command only: never in the URL, .git/config, or error output.
b64 = base64.b64encode(f"x-access-token:{gh}".encode()).decode()
auth = ["-c", f"http.extraHeader=Authorization: Basic {b64}"]
def git(*args, cwd=None):
    r = subprocess.run(["git", *auth, *args], cwd=cwd, capture_output=True, text=True)
    if r.returncode != 0:
        msg = (r.stderr or r.stdout).replace(gh, "***")
        raise RuntimeError(f"git {args[0]} failed: {msg.strip()}")
    return r.stdout

os.chdir(WORK)
repo_dir = os.path.join(WORK, "mcp-poison-detect")
if os.path.isdir(os.path.join(repo_dir, ".git")):
    git("fetch", "-q", cwd=repo_dir); git("checkout", "-q", BRANCH, cwd=repo_dir); git("pull", "-q", cwd=repo_dir)
else:
    git("clone", "-q", "-b", BRANCH, f"https://{REPO}.git", repo_dir)
os.chdir(repo_dir)
print("commit:", git("log", "-1", "--oneline", cwd=repo_dir))
assert os.path.exists(MIN_COMMIT_FILE), f"{MIN_COMMIT_FILE} missing: push the latest commit first (needs results/wb3_*)"


In [ ]:
!pip install -q -r requirements.txt
if GPU_DEPS:
    !pip install -q -r requirements-gpu.txt
!pip install -q -e . --no-deps


In [ ]:
import torch
!nvidia-smi --query-gpu=name,memory.total --format=csv
assert torch.cuda.device_count() >= 2, "Need GPU T4 x2"
!python scripts/adv_train.py --tag adv2_final --set 'advtrain.sources={SRC_TRAIN}' 'advtrain.seeds=[42]'


In [ ]:
import glob, os
adv_run = sorted(glob.glob("/kaggle/working/outputs/adv_train/*"))[-1]
ckpt = os.path.join(adv_run, "ckpt_adv")
assert os.path.exists(os.path.join(ckpt, "detector.json")), ckpt
!python scripts/attack_whitebox.py --tag wb5_final_cal_b30 --set whitebox.detector_ckpt={ckpt} {ATTACK}


In [ ]:
!python scripts/calibrate_adv.py --tag cal_adv2_s43 --set 'caladv.adv_sources={SRC_CAL}' caladv.seed=43
!python scripts/calibrate_adv.py --tag cal_adv2_s44 --set 'caladv.adv_sources={SRC_CAL}' caladv.seed=44


In [ ]:
import glob, json
run = sorted(glob.glob("/kaggle/working/outputs/attack_whitebox/*"))[-1]
s = json.load(open(f"{run}/summary.json"))
print("FINAL-SYSTEM ATTACK:", os.path.basename(run), "| calibrated", s.get("calibrated"), "| evaded", s["n_evaded"],
      "| genuine", s.get("n_genuine_evasions"), "| of", s["n_items"])
for r in sorted(glob.glob("/kaggle/working/outputs/calibrate_adv/*")):
    print(open(f"{r}/cal_adv.md").read().split("## Target 5%")[0])
